# Exercise 27 - Titanic interpolation

In this exercise, we will fill in missing data from the famous Titanic data set: a table of all passengers on that famous, doomed ship. Many of the columns in this file are complete, but some are missing data. It will be up to you to decide whether and how to fill in that missing data.

For this exercise, I would like you to do the following:

1. Load the ```titanic3.xls``` data into a data frame. Note that this file is an Excel spreadsheet, so you won’t be able to use ```read_csv```. Rather, you’ll have to use ```read_excel```.

In [25]:
import pandas as pd

titanic_data = pd.read_excel(
    "./titanic3.xls",
    header=0,
)

titanic_data.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


2. Determine which columns contain null values.

In [26]:
titanic_data.isna().sum()

pclass          0
survived        0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

In [27]:
titanic_data.columns[titanic_data.isna().sum() > 0]

Index(['age', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest'], dtype='str')

3. For each column containing null values, decide whether you will fill it with a value—and if so, with what value, calculated or otherwise.

**Answer:**

* ```fare``` and ```embarked``` have up to two ```NaN``` values. Let's remove them.
* We can replace ```NaN``` values in the ```age``` column with the mean.
* We can replace ```NaN``` values in the ```home.dest``` column with the mode (the most common value).

In [28]:
titanic_data = titanic_data.dropna(subset=["fare", "embarked"]) # delete rows with NaN in fare and embarked columns
titanic_data["age"] = titanic_data["age"].fillna( # replace NaN values in age column with the mean age
    titanic_data["age"].mean()
)

In [33]:
titanic_data_copy = titanic_data.copy() # copy of the data to be use in "Beyond the exercise" section

titanic_data["home.dest"] = titanic_data["home.dest"].fillna( # replace NaN values in homes.dest column with the mode
    titanic_data["home.dest"].mode()[0]
)

## Beyond the exercise

In these tasks, we will do something I mentioned earlier: replace ```NaN``` values in the ```home.dest``` column with the most common value from that person’s ```embarked``` column. This will take several steps:

1. Create a series (```most_common_destinations```) in which the index contains the unique values from the ```embarked``` column and the values are the most common destination for each value of ```embarked```.

In [36]:
embarkment_sites =titanic_data_copy["embarked"].values.unique() # embarkment sites of titanic
destinations = []
for embarked_site in embarkment_sites:
    destinations.append(
        # the most common destination from embarkment_site
        titanic_data_copy.loc[titanic_data_copy["embarked"] == embarked_site, "home.dest"].mode()[0]
    )

most_common_destinations = pd.Series(
    index=embarkment_sites,
    data=destinations,
)

most_common_destinations


S           New York, NY
C           New York, NY
Q    Ireland Chicago, IL
dtype: str

2. Replace ```NaN``` values in the ```home.dest``` column with values from embarked. (Because values in ```embarked``` and ```home.dest``` are distinct, this is an OK middle step.)

In [37]:
titanic_data_copy["home.dest"] = titanic_data_copy["home.dest"].fillna(
    titanic_data_copy["embarked"]
)

3. Use the ```most_common_destinations``` series to replace values in ```home.dest``` with the most common values for each embarkation point.

In [38]:
titanic_data_copy = titanic_data_copy["home.dest"].replace(most_common_destinations)